# Train LSTM2 on Colab

This notebook sets up Colab, copies a selected `data/lstm2_processed/<dataset_version>` dataset to local disk, and launches a configurable LSTM2 training module.

## 1. Mount Drive

In [ ]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

## 2. Paths

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# Code should live on local Colab disk. Drive is only for large data/checkpoints/logs.
# Set this to your GitHub repo URL before running the cell.
GITHUB_REPO_URL = "https://github.com/sungjelly/Seoul_bike_project.git"
GITHUB_BRANCH = "main"
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Seoul_bike_project")

# Change these values to switch LSTM2 experiments.
DATASET_VERSION = "tts_lstm2"
MODEL_VERSION = "tts_lstm2"
BASE_DATASET_NAME = "base"
PROCESSED_DIR_NAME = "lstm2_processed"
ARTIFACT_FAMILY = "lstm2"
CONFIG_FAMILY = "lstm2"
TRAIN_MODULE = "src.training.lstm2_training.train_lstm2"
CONFIG_PATH = f"configs/models/{CONFIG_FAMILY}/{MODEL_VERSION}.yaml"
BATCH_SIZE = 32768
MAX_EPOCHS = None
SMOKE_TEST = False

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(CODE_DIR)],
        check=True,
    )

PROJECT_DIR = CODE_DIR
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / "data" / PROCESSED_DIR_NAME
LOCAL_PROCESSED_ROOT = Path("/content") / PROCESSED_DIR_NAME
DRIVE_DATA_DIR = DRIVE_PROCESSED_ROOT / DATASET_VERSION
DRIVE_BASE_DATA_DIR = DRIVE_PROCESSED_ROOT / BASE_DATASET_NAME
LOCAL_DATA_DIR = LOCAL_PROCESSED_ROOT / DATASET_VERSION
LOCAL_BASE_DATA_DIR = LOCAL_PROCESSED_ROOT / BASE_DATASET_NAME
DATA_DIR = LOCAL_DATA_DIR

CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / ARTIFACT_FAMILY / MODEL_VERSION
MODEL_DIR = DRIVE_ROOT / "models" / ARTIFACT_FAMILY / MODEL_VERSION
LOG_DIR = DRIVE_ROOT / "logs" / ARTIFACT_FAMILY / MODEL_VERSION

os.chdir(PROJECT_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Code directory:", PROJECT_DIR)
print("Drive artifact root:", DRIVE_ROOT)
print("Drive dataset:", DRIVE_DATA_DIR)
print("Drive base dataset:", DRIVE_BASE_DATA_DIR)
print("Local dataset:", DATA_DIR)
print("Training module:", TRAIN_MODULE)
print("Config path:", CONFIG_PATH)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Current working directory:", Path.cwd())

## 3. Install Requirements

In [ ]:
%cd /content/Seoul_bike_project
!pip install -r requirements.txt

## 4. Copy Dataset to Colab Disk

In [ ]:
import shutil

required_drive_dirs = [DRIVE_BASE_DATA_DIR, DRIVE_DATA_DIR]
missing = [str(path) for path in required_drive_dirs if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Drive LSTM2 dataset directories. Upload data/lstm2_processed/base "
        "and data/lstm2_processed/tts_lstm2 before running this cell: " + str(missing)
    )

LOCAL_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
for src, dst in [(DRIVE_BASE_DATA_DIR, LOCAL_BASE_DATA_DIR), (DRIVE_DATA_DIR, LOCAL_DATA_DIR)]:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Copied {src} -> {dst}")

print("Using local dataset:", DATA_DIR)

## 5. Verify Dataset

In [ ]:
import json
import numpy as np

array_dir = LOCAL_BASE_DATA_DIR
dynamic = np.load(array_dir / "dynamic_features.npy", mmap_mode="r")
target_time = np.load(array_dir / "target_time_features.npy", mmap_mode="r")
targets = np.load(array_dir / "targets.npy", mmap_mode="r")
targets_raw = np.load(array_dir / "targets_raw.npy", mmap_mode="r")
feature_config = json.loads((LOCAL_DATA_DIR / "feature_config.json").read_text())

assert dynamic.ndim == 3 and dynamic.shape[-1] == 7, dynamic.shape
assert target_time.ndim == 2 and target_time.shape[-1] == 8, target_time.shape
assert targets.ndim == 4 and targets.shape[-2:] == (8, 2), targets.shape
assert targets_raw.ndim == 4 and targets_raw.shape[-2:] == (8, 2), targets_raw.shape
assert len(feature_config["dynamic_feature_columns"]) == 7
assert len(feature_config["target_time_feature_columns"]) == 8
for excluded in ["avg_duration_min", "avg_distance_m"]:
    assert excluded not in feature_config["dynamic_feature_columns"]
    assert excluded not in feature_config["target_time_feature_columns"]
assert (LOCAL_DATA_DIR / "sample_index_train.npy").exists()
assert (LOCAL_DATA_DIR / "sample_index_val.npy").exists()

print("dynamic_features:", dynamic.shape)
print("target_time_features:", target_time.shape)
print("targets:", targets.shape)
print("targets_raw:", targets_raw.shape)
print("dynamic columns:", feature_config["dynamic_feature_columns"])
print("target time columns:", feature_config["target_time_feature_columns"])

## 6. W&B Login

In [ ]:
import os

try:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
except Exception:
    pass

if os.environ.get("WANDB_API_KEY"):
    !wandb login $WANDB_API_KEY
else:
    print("WANDB_API_KEY not found; training can still run if wandb is disabled in config or CLI.")

## 7. Train With Source Script

In [ ]:
extra_args = []
if MAX_EPOCHS is not None:
    extra_args += ["--max_epochs", str(MAX_EPOCHS)]
if SMOKE_TEST:
    extra_args += ["--smoke_test", "true"]
else:
    extra_args += ["--smoke_test", "false"]

cmd = [
    "python", "-m", TRAIN_MODULE,
    "--config", CONFIG_PATH,
    "--data_dir", str(DATA_DIR),
    "--checkpoint_dir", str(CHECKPOINT_DIR),
    "--model_dir", str(MODEL_DIR),
    "--log_dir", str(LOG_DIR),
    "--batch_size", str(BATCH_SIZE),
] + extra_args
print(" ".join(cmd))
subprocess.run(cmd, check=True)